<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/01_generate_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ MetricGuard AI — Synthetic Analytics Dataset

## Phase 1 — Building the Northstar Commerce Analytics Universe

This notebook generates the synthetic knowledge base used by MetricGuard AI.

The dataset represents a fictional e-commerce organization called
**Northstar Commerce**.

The environment intentionally contains:

- multiple analytics datasets
- SQL transformations
- dbt-style YAML documentation
- dashboard definitions
- metric definitions
- metric version history
- business-rule changes
- incident tickets
- analyst notes
- stale logic
- conflicting definitions
- known lineage relationships

Some inconsistencies are intentionally introduced so that MetricGuard AI
can later detect, investigate, and explain them.

## Company — Northstar Commerce

Northstar Commerce is a fictional omnichannel e-commerce company.

The organization contains several teams that consume analytics differently:

- Executive Leadership
- Finance
- Marketing
- Growth
- Operations
- Data & Analytics

Each team uses dashboards and metrics for different business decisions.

Over time, business definitions have changed but not every dashboard,
SQL transformation, or documentation source has been updated consistently.

This creates situations where multiple systems use the same metric name
while applying different underlying business logic.

## Raw Operational Data

Northstar Commerce operates using six primary raw datasets.

| Dataset | Purpose |
|---|---|
| `raw_customers` | Customer profiles and acquisition information |
| `raw_orders` | Order-level transactions |
| `raw_order_items` | Products contained within each order |
| `raw_payments` | Payments and chargeback transactions |
| `raw_refunds` | Customer refund transactions |
| `raw_web_sessions` | Website and mobile browsing sessions |

These datasets represent the source systems that feed Northstar's
analytics warehouse.

All records are synthetic and generated deterministically for
reproducibility.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

START_DATE = pd.Timestamp("2025-01-01")
END_DATE = pd.Timestamp("2026-07-31")

N_CUSTOMERS = 2_000
N_ORDERS = 8_000
N_SESSIONS = 20_000

print("Northstar Commerce Dataset Configuration")
print("-" * 50)
print(f"Customers : {N_CUSTOMERS:,}")
print(f"Orders    : {N_ORDERS:,}")
print(f"Sessions  : {N_SESSIONS:,}")
print(f"Period    : {START_DATE.date()} → {END_DATE.date()}")
print(f"Seed      : {SEED}")

Northstar Commerce Dataset Configuration
--------------------------------------------------
Customers : 2,000
Orders    : 8,000
Sessions  : 20,000
Period    : 2025-01-01 → 2026-07-31
Seed      : 42


In [2]:
def random_dates(start, end, n, rng):
    """
    Generate n random dates between start and end.
    """
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    total_days = (end - start).days

    offsets = rng.integers(
        0,
        total_days + 1,
        size=n
    )

    return start + pd.to_timedelta(offsets, unit="D")

### Generate raw_customers

In [3]:
customer_ids = [
    f"CUST-{i:05d}"
    for i in range(1, N_CUSTOMERS + 1)
]

signup_dates = random_dates(
    START_DATE - pd.Timedelta(days=365),
    END_DATE,
    N_CUSTOMERS,
    rng
)

raw_customers = pd.DataFrame({
    "customer_id": customer_ids,

    "signup_date": signup_dates,

    "region": rng.choice(
        ["North", "South", "East", "West", "Central"],
        size=N_CUSTOMERS,
        p=[0.18, 0.24, 0.20, 0.23, 0.15]
    ),

    "acquisition_channel": rng.choice(
        [
            "organic",
            "paid_search",
            "social",
            "email",
            "affiliate",
            "referral"
        ],
        size=N_CUSTOMERS,
        p=[0.25, 0.22, 0.18, 0.12, 0.10, 0.13]
    ),

    "customer_segment": rng.choice(
        ["new", "regular", "high_value"],
        size=N_CUSTOMERS,
        p=[0.35, 0.50, 0.15]
    ),

    "customer_status": rng.choice(
        ["active", "inactive"],
        size=N_CUSTOMERS,
        p=[0.88, 0.12]
    )
})

raw_customers.head()

,customer_id,signup_date,region,acquisition_channel,customer_segment,customer_status
0,CUST-00001,2024-03-26,North,affiliate,new,active
1,CUST-00002,2025-12-31,East,social,regular,inactive
2,CUST-00003,2025-09-09,North,organic,regular,active
3,CUST-00004,2025-02-18,North,email,new,active
4,CUST-00005,2025-02-12,West,organic,new,inactive


In [4]:
raw_customers.shape

(2000, 6)

### Generate raw_orders

In [5]:
order_ids = [
    f"ORD-{i:06d}"
    for i in range(1, N_ORDERS + 1)
]

sampled_customer_indices = rng.integers(
    0,
    N_CUSTOMERS,
    size=N_ORDERS
)

sampled_customers = (
    raw_customers
    .iloc[sampled_customer_indices]
    .reset_index(drop=True)
)

order_dates = []

for signup_date in sampled_customers["signup_date"]:

    earliest_order_date = max(
        pd.Timestamp(signup_date),
        START_DATE
    )

    days_available = (
        END_DATE - earliest_order_date
    ).days

    random_offset = int(
        rng.integers(0, days_available + 1)
    )

    order_dates.append(
        earliest_order_date +
        pd.Timedelta(days=random_offset)
    )

raw_orders = pd.DataFrame({
    "order_id": order_ids,

    "customer_id":
        sampled_customers["customer_id"].values,

    "order_date":
        pd.to_datetime(order_dates),

    "order_status": rng.choice(
        [
            "completed",
            "shipped",
            "cancelled",
            "returned",
            "pending"
        ],
        size=N_ORDERS,
        p=[0.65, 0.12, 0.10, 0.08, 0.05]
    ),

    "sales_channel": rng.choice(
        ["web", "mobile_app"],
        size=N_ORDERS,
        p=[0.64, 0.36]
    ),

    "currency": "INR"
})

raw_orders.head()

,order_id,customer_id,order_date,order_status,sales_channel,currency
0,ORD-000001,CUST-01455,2025-09-25,completed,web,INR
1,ORD-000002,CUST-01748,2025-08-11,completed,mobile_app,INR
2,ORD-000003,CUST-00287,2025-01-17,completed,mobile_app,INR
3,ORD-000004,CUST-00493,2026-05-24,completed,mobile_app,INR
4,ORD-000005,CUST-00075,2026-07-07,pending,web,INR


### Generate raw_order_items

In [6]:
CATEGORY_PRICE_RANGES = {
    "electronics": (800, 12000),
    "home": (300, 5000),
    "beauty": (150, 2500),
    "fitness": (250, 6000),
    "accessories": (100, 3500)
}

item_rows = []

item_counter = 1

for _, order in raw_orders.iterrows():

    number_of_items = int(
        rng.integers(1, 5)
    )

    for _ in range(number_of_items):

        category = rng.choice(
            list(CATEGORY_PRICE_RANGES.keys())
        )

        minimum_price, maximum_price = (
            CATEGORY_PRICE_RANGES[category]
        )

        quantity = int(
            rng.integers(1, 4)
        )

        unit_price = round(
            float(
                rng.uniform(
                    minimum_price,
                    maximum_price
                )
            ),
            2
        )

        line_gross_amount = round(
            quantity * unit_price,
            2
        )

        discount_rate = float(
            rng.choice(
                [0, 0.05, 0.10, 0.15, 0.20],
                p=[0.45, 0.15, 0.18, 0.12, 0.10]
            )
        )

        line_discount_amount = round(
            line_gross_amount * discount_rate,
            2
        )

        line_net_amount = round(
            line_gross_amount -
            line_discount_amount,
            2
        )

        product_number = int(
            rng.integers(1, 301)
        )

        item_rows.append({
            "order_item_id":
                f"ITEM-{item_counter:07d}",

            "order_id":
                order["order_id"],

            "product_id":
                f"PROD-{product_number:04d}",

            "product_category":
                category,

            "quantity":
                quantity,

            "unit_price":
                unit_price,

            "line_gross_amount":
                line_gross_amount,

            "line_discount_amount":
                line_discount_amount,

            "line_net_amount":
                line_net_amount
        })

        item_counter += 1

raw_order_items = pd.DataFrame(item_rows)

raw_order_items.head()

,order_item_id,order_id,product_id,product_category,quantity,unit_price,line_gross_amount,line_discount_amount,line_net_amount
0,ITEM-0000001,ORD-000001,PROD-0013,accessories,3,1286.49,3859.47,385.95,3473.52
1,ITEM-0000002,ORD-000001,PROD-0188,electronics,2,11805.83,23611.66,0.00,23611.66
2,ITEM-0000003,ORD-000002,PROD-0242,accessories,1,796.40,796.40,39.82,756.58
3,ITEM-0000004,ORD-000002,PROD-0125,accessories,2,1098.15,2196.30,0.00,2196.30
4,ITEM-0000005,ORD-000002,PROD-0146,electronics,1,10188.22,10188.22,2037.64,8150.58


In [7]:
raw_order_items.shape

(20115, 9)

### Generate Final raw_orders

In [8]:
order_financials = (
    raw_order_items
    .groupby("order_id")
    .agg(
        gross_merchandise_amount=(
            "line_gross_amount",
            "sum"
        ),

        discount_amount=(
            "line_discount_amount",
            "sum"
        ),

        subtotal_amount=(
            "line_net_amount",
            "sum"
        )
    )
    .reset_index()
)

raw_orders = raw_orders.merge(
    order_financials,
    on="order_id",
    how="left"
)

raw_orders["shipping_amount"] = np.where(
    raw_orders["subtotal_amount"] >= 2000,
    0,
    rng.choice(
        [49, 79, 99],
        size=len(raw_orders)
    )
)

raw_orders["tax_amount"] = (
    raw_orders["subtotal_amount"] * 0.18
).round(2)

raw_orders["order_total_amount"] = (
    raw_orders["subtotal_amount"]
    + raw_orders["shipping_amount"]
    + raw_orders["tax_amount"]
).round(2)

raw_orders.head()

,order_id,customer_id,order_date,order_status,sales_channel,currency,gross_merchandise_amount,discount_amount,subtotal_amount,shipping_amount,tax_amount,order_total_amount
0,ORD-000001,CUST-01455,2025-09-25,completed,web,INR,27471.13,385.95,27085.18,0,4875.33,31960.51
1,ORD-000002,CUST-01748,2025-08-11,completed,mobile_app,INR,16962.29,2266.53,14695.76,0,2645.24,17341.00
2,ORD-000003,CUST-00287,2025-01-17,completed,mobile_app,INR,55156.31,3277.06,51879.25,0,9338.26,61217.51
3,ORD-000004,CUST-00493,2026-05-24,completed,mobile_app,INR,14068.62,394.88,13673.74,0,2461.27,16135.01
4,ORD-000005,CUST-00075,2026-07-07,pending,web,INR,6878.37,687.84,6190.53,0,1114.30,7304.83


### Generate raw_payments

In [9]:
payment_rows = []

payment_counter = 1

for _, order in raw_orders.iterrows():

    status = order["order_status"]

    if status in ["completed", "shipped", "returned"]:

        transaction_status = "successful"

    elif status == "pending":

        transaction_status = "pending"

    else:

        transaction_status = "failed"

    payment_rows.append({
        "payment_id":
            f"PAY-{payment_counter:07d}",

        "order_id":
            order["order_id"],

        "transaction_date":
            order["order_date"],

        "transaction_type":
            "payment",

        "payment_method":
            rng.choice(
                [
                    "credit_card",
                    "debit_card",
                    "upi",
                    "wallet",
                    "net_banking"
                ]
            ),

        "transaction_amount":
            order["order_total_amount"],

        "transaction_status":
            transaction_status
    })

    payment_counter += 1


eligible_chargebacks = raw_orders[
    raw_orders["order_status"].isin(
        ["completed", "shipped", "returned"]
    )
].sample(
    frac=0.015,
    random_state=SEED
)


for _, order in eligible_chargebacks.iterrows():

    chargeback_amount = round(
        float(
            order["order_total_amount"]
            * rng.uniform(0.50, 1.00)
        ),
        2
    )

    chargeback_date = min(
        order["order_date"]
        + pd.Timedelta(
            days=int(rng.integers(7, 46))
        ),
        END_DATE
    )

    payment_rows.append({
        "payment_id":
            f"PAY-{payment_counter:07d}",

        "order_id":
            order["order_id"],

        "transaction_date":
            chargeback_date,

        "transaction_type":
            "chargeback",

        "payment_method":
            "original_payment_method",

        "transaction_amount":
            chargeback_amount,

        "transaction_status":
            "posted"
    })

    payment_counter += 1


raw_payments = pd.DataFrame(payment_rows)

raw_payments.head()

,payment_id,order_id,transaction_date,transaction_type,payment_method,transaction_amount,transaction_status
0,PAY-0000001,ORD-000001,2025-09-25,payment,credit_card,31960.51,successful
1,PAY-0000002,ORD-000002,2025-08-11,payment,debit_card,17341.00,successful
2,PAY-0000003,ORD-000003,2025-01-17,payment,credit_card,61217.51,successful
3,PAY-0000004,ORD-000004,2026-05-24,payment,debit_card,16135.01,successful
4,PAY-0000005,ORD-000005,2026-07-07,payment,credit_card,7304.83,pending


In [10]:
raw_payments["transaction_type"].value_counts()

,count
transaction_type,
payment,8000
chargeback,101


### Generate raw_refunds

In [11]:
refund_candidates = raw_orders[
    raw_orders["order_status"].isin(
        ["returned", "completed"]
    )
].copy()

returned_orders = refund_candidates[
    refund_candidates["order_status"] == "returned"
]

additional_refunds = refund_candidates[
    refund_candidates["order_status"] == "completed"
].sample(
    frac=0.04,
    random_state=SEED
)

refund_orders = pd.concat(
    [
        returned_orders,
        additional_refunds
    ]
).drop_duplicates(
    subset=["order_id"]
)


refund_rows = []

for i, (_, order) in enumerate(
    refund_orders.iterrows(),
    start=1
):

    refund_fraction = float(
        rng.uniform(0.20, 1.00)
    )

    refund_amount = round(
        order["subtotal_amount"]
        * refund_fraction,
        2
    )

    refund_date = min(
        order["order_date"]
        + pd.Timedelta(
            days=int(rng.integers(1, 31))
        ),
        END_DATE
    )

    refund_rows.append({
        "refund_id":
            f"REF-{i:06d}",

        "order_id":
            order["order_id"],

        "refund_date":
            refund_date,

        "refund_amount":
            refund_amount,

        "refund_reason":
            rng.choice(
                [
                    "damaged_item",
                    "customer_return",
                    "wrong_item",
                    "late_delivery",
                    "quality_issue",
                    "other"
                ]
            ),

        "refund_status":
            rng.choice(
                ["completed", "pending"],
                p=[0.92, 0.08]
            )
    })


raw_refunds = pd.DataFrame(refund_rows)

raw_refunds.head()

,refund_id,order_id,refund_date,refund_amount,refund_reason,refund_status
0,REF-000001,ORD-000028,2025-12-07,13062.71,late_delivery,completed
1,REF-000002,ORD-000059,2026-06-08,9490.45,other,completed
2,REF-000003,ORD-000066,2026-07-09,18572.42,quality_issue,completed
3,REF-000004,ORD-000085,2026-03-29,5889.54,customer_return,completed
4,REF-000005,ORD-000086,2025-10-29,3770.85,other,completed


In [12]:
raw_refunds.shape

(845, 6)

### Generate raw_web_sessions

In [13]:
session_ids = [
    f"SES-{i:07d}"
    for i in range(1, N_SESSIONS + 1)
]

session_dates = random_dates(
    START_DATE,
    END_DATE,
    N_SESSIONS,
    rng
)

session_hours = rng.integers(
    0,
    24,
    size=N_SESSIONS
)

session_minutes = rng.integers(
    0,
    60,
    size=N_SESSIONS
)

session_start = (
    session_dates
    + pd.to_timedelta(
        session_hours,
        unit="h"
    )
    + pd.to_timedelta(
        session_minutes,
        unit="m"
    )
)

known_customer = (
    rng.random(N_SESSIONS) < 0.65
)

session_customers = np.where(
    known_customer,
    rng.choice(
        raw_customers["customer_id"],
        size=N_SESSIONS
    ),
    None
)

started_checkout = (
    rng.random(N_SESSIONS) < 0.24
)

completed_purchase = (
    started_checkout
    & (rng.random(N_SESSIONS) < 0.48)
)

raw_web_sessions = pd.DataFrame({
    "session_id":
        session_ids,

    "customer_id":
        session_customers,

    "session_start":
        session_start,

    "traffic_source":
        rng.choice(
            [
                "organic_search",
                "paid_search",
                "social",
                "email",
                "direct",
                "affiliate"
            ],
            size=N_SESSIONS,
            p=[
                0.25,
                0.22,
                0.18,
                0.12,
                0.15,
                0.08
            ]
        ),

    "device_type":
        rng.choice(
            [
                "mobile",
                "desktop",
                "tablet"
            ],
            size=N_SESSIONS,
            p=[0.62, 0.32, 0.06]
        ),

    "started_checkout":
        started_checkout,

    "completed_purchase":
        completed_purchase,

    "order_id":
        None
})

In [14]:
purchase_session_indices = (
    raw_web_sessions[
        raw_web_sessions["completed_purchase"]
    ].index
)

linked_orders = raw_orders.sample(
    n=len(purchase_session_indices),
    replace=True,
    random_state=SEED
).reset_index(drop=True)

for position, session_index in enumerate(
    purchase_session_indices
):

    order = linked_orders.iloc[position]

    raw_web_sessions.loc[
        session_index,
        "order_id"
    ] = order["order_id"]

    raw_web_sessions.loc[
        session_index,
        "customer_id"
    ] = order["customer_id"]

raw_web_sessions.head()

,session_id,customer_id,session_start,traffic_source,device_type,started_checkout,completed_purchase,order_id
0,SES-0000001,CUST-00710,2026-01-23 21:11:00,organic_search,desktop,False,False,None
1,SES-0000002,CUST-00497,2026-03-25 09:34:00,organic_search,mobile,False,False,None
2,SES-0000003,CUST-00768,2026-02-16 17:12:00,email,mobile,False,False,None
3,SES-0000004,None,2025-10-11 17:43:00,email,mobile,False,False,None
4,SES-0000005,CUST-01665,2026-01-12 23:33:00,affiliate,desktop,True,False,None


### Check Raw Data

In [15]:
datasets = {
    "raw_customers": raw_customers,
    "raw_orders": raw_orders,
    "raw_order_items": raw_order_items,
    "raw_payments": raw_payments,
    "raw_refunds": raw_refunds,
    "raw_web_sessions": raw_web_sessions
}

summary = pd.DataFrame({
    "dataset": datasets.keys(),
    "rows": [
        len(df)
        for df in datasets.values()
    ],
    "columns": [
        len(df.columns)
        for df in datasets.values()
    ]
})

summary

,dataset,rows,columns
0,raw_customers,2000,6
1,raw_orders,8000,12
2,raw_order_items,20115,9
3,raw_payments,8101,7
4,raw_refunds,845,6
5,raw_web_sessions,20000,8


In [16]:
print("Northstar Commerce — Data Quality Checks")
print("=" * 55)

checks = {
    "Customer IDs unique":
        raw_customers["customer_id"].is_unique,

    "Order IDs unique":
        raw_orders["order_id"].is_unique,

    "Order item IDs unique":
        raw_order_items["order_item_id"].is_unique,

    "Payment IDs unique":
        raw_payments["payment_id"].is_unique,

    "Refund IDs unique":
        raw_refunds["refund_id"].is_unique,

    "All order customers exist":
        raw_orders["customer_id"]
        .isin(raw_customers["customer_id"])
        .all(),

    "All order items reference valid orders":
        raw_order_items["order_id"]
        .isin(raw_orders["order_id"])
        .all(),

    "All payments reference valid orders":
        raw_payments["order_id"]
        .isin(raw_orders["order_id"])
        .all(),

    "All refunds reference valid orders":
        raw_refunds["order_id"]
        .isin(raw_orders["order_id"])
        .all(),

    "No negative order totals":
        (raw_orders["order_total_amount"] >= 0)
        .all(),

    "No negative refunds":
        (raw_refunds["refund_amount"] >= 0)
        .all()
}

for check_name, result in checks.items():

    icon = "✅" if result else "❌"

    print(
        f"{icon} {check_name}: {result}"
    )

Northstar Commerce — Data Quality Checks
✅ Customer IDs unique: True
✅ Order IDs unique: True
✅ Order item IDs unique: True
✅ Payment IDs unique: True
✅ Refund IDs unique: True
✅ All order customers exist: True
✅ All order items reference valid orders: True
✅ All payments reference valid orders: True
✅ All refunds reference valid orders: True
✅ No negative order totals: True
✅ No negative refunds: True


## Export CSVs

In [17]:
EXPORT_DIR = Path(
    "/content/metricguard_raw_tabular"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for dataset_name, dataframe in datasets.items():

    file_path = (
        EXPORT_DIR /
        f"{dataset_name}.csv"
    )

    dataframe.to_csv(
        file_path,
        index=False
    )

    print(
        f"Saved: {file_path}"
    )

Saved: /content/metricguard_raw_tabular/raw_customers.csv
Saved: /content/metricguard_raw_tabular/raw_orders.csv
Saved: /content/metricguard_raw_tabular/raw_order_items.csv
Saved: /content/metricguard_raw_tabular/raw_payments.csv
Saved: /content/metricguard_raw_tabular/raw_refunds.csv
Saved: /content/metricguard_raw_tabular/raw_web_sessions.csv


In [18]:
list(EXPORT_DIR.iterdir())

[PosixPath('/content/metricguard_raw_tabular/raw_payments.csv'),
 PosixPath('/content/metricguard_raw_tabular/raw_web_sessions.csv'),
 PosixPath('/content/metricguard_raw_tabular/raw_orders.csv'),
 PosixPath('/content/metricguard_raw_tabular/raw_refunds.csv'),
 PosixPath('/content/metricguard_raw_tabular/raw_order_items.csv'),
 PosixPath('/content/metricguard_raw_tabular/raw_customers.csv')]

In [19]:
import shutil

archive_path = shutil.make_archive(
    "/content/metricguard_raw_tabular",
    "zip",
    EXPORT_DIR
)

print(archive_path)

/content/metricguard_raw_tabular.zip


In [20]:
from google.colab import files

files.download(
    "/content/metricguard_raw_tabular.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>